# Visualizing Pt–Water Molecular Dynamics with MACE


## Goal of the notebook

Let us create a short Pt–water molecular dynamics trajectory with a pretrained MACE model. We will turn its saved structures into an animation and inspect the interface.


## Description of the method

MACE predicts atomic energies and forces within ASE. We will constrain the Pt region, assign thermal velocities, and run dynamics with a Berendsen thermostat. A fixed camera will render each saved frame, and imageio will combine the frames into a GIF.

This short calculation introduces the workflow and its choices. Before we draw quantitative conclusions, we should check the model, trajectory length, and thermostat behavior.


## What we will do

1. Let us inspect the Pt–water structure and identify fixed and mobile atoms.
2. We will set the temperature, time step, and thermostat, attach MACE, and save a trajectory.
3. We will render the frames and assemble an animation of the interface.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q mace-torch ase imageio pillow matplotlib; fi

In [ ]:
import imageio
import matplotlib.pyplot as plt
import os
import pandas as pd

from ase import units
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.io.trajectory import Trajectory
from ase.md.nvtberendsen import NVTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.visualize import view
from IPython.display import display
from IPython.display import Image as IPImage
from mace.calculators import MACECalculator
from pathlib import Path
from PIL import Image

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

## Choose the trajectory view

Set `animation_mode` to `"static"` for a presentation slide or `"animated"` to view the trajectory as a GIF.


In [ ]:
animation_mode = "static"  # Choose "animated" or "static".

In [ ]:
# Download once into the current folder.
!if [ ! -s Pt-32H2O_MD.xyz ]; then wget -q https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Pt-w/Pt-32H2O_MD.xyz -O Pt-32H2O_MD.xyz; fi
if not Path('Pt-32H2O_MD.xyz').is_file() or Path('Pt-32H2O_MD.xyz').stat().st_size == 0:
    raise FileNotFoundError("Download failed: Pt-32H2O_MD.xyz")

## Load and inspect the model

The `.xyz` file stores a Pt slab with 32 water molecules above it. Here the adsorbate layer contains water. Let us inspect the structure before the simulation.


In [ ]:
atoms = read("Pt-32H2O_MD.xyz")
view(atoms, viewer="x3d")

## Choose molecular-dynamics settings

The target temperature is 300 K. We use a 0.5 fs time step to resolve the fast O–H vibrations near 3,500 cm⁻¹. A smaller step helps integrate this motion accurately and keeps the trajectory stable.


> **Pause and predict:** What could happen if the time step is too large for O–H stretching?
>
> **What to look for:** Check trajectory stability before you turn the frames into an animation.


In [ ]:
number_of_md_steps = 100
temperature_k = 300.0
# A 0.5 fs step resolves the high-frequency O–H motion.
time_step_fs = 0.5

This model freezes every Pt atom, so only water molecules move. The SpookyNet example also keeps the Pt slab fixed. A rigid slab suits a question about water motion while surface reconstruction stays outside the model. The calculator may still evaluate the full structure at each step.


In [ ]:
# Tag the Pt substrate, then keep every tagged Pt atom fixed.
for atom in atoms: atom.tag = 42 if atom.symbol == "Pt" else 0
fixed_pt_indices = [atom.index for atom in atoms if atom.tag == 42]
atoms.set_constraint(FixAtoms(indices=fixed_pt_indices))

## Configure MACE

The `mace_agnesi_small.model` file is a pretrained MACE-MP-0 foundation model from the [ACEsuit GitHub release](https://github.com/ACEsuit/mace-mp/releases/tag/mace_mp_0b). Its broad DFT training set supports transfer across many chemistries. We still need to check its suitability for this interface. The setting `use_cu_eq=False` disables a CUDA-specific equivariance optimization for compatibility. The float64 message reflects MACE's default precision, which matches the model training dtype.


In [ ]:
# Download once into the current folder.
!if [ ! -s mace_agnesi_small.model ]; then wget -q https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b/mace_agnesi_small.model -O mace_agnesi_small.model; fi
if not Path('mace_agnesi_small.model').is_file() or Path('mace_agnesi_small.model').stat().st_size == 0:
    raise FileNotFoundError("Download failed: mace_agnesi_small.model")

## Run NVT molecular dynamics

Let us identify the main settings:

- `NVTBerendsen` rescales velocities toward the target temperature. `taut_fs = 500 fs` sets the coupling time; this longer time gives a gentle correction.
- `MaxwellBoltzmannDistribution` assigns initial velocities at 300 K.
- `interval=1` saves every step, so this 100-step run has full frame resolution.


In [ ]:
model_path = "mace_agnesi_small.model"
atoms.calc = MACECalculator(model_paths=model_path, use_cu_eq=False)
thermostat_time_constant_fs = 500.0
dynamics = NVTBerendsen(atoms, time_step_fs * units.fs, temperature_k, thermostat_time_constant_fs * units.fs)

In [ ]:
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_k)
trajectory = Trajectory("Model.traj", "w", atoms)
dynamics.attach(trajectory.write, interval=1)
dynamics.run(number_of_md_steps)

### Read the simulated time

Compare the physical duration and target temperature with the movie. GIF playback speed describes display time; the time step and number of integration steps set simulation time.


In [ ]:
print("Simulation duration / fs:", number_of_md_steps * time_step_fs)
print("Final kinetic temperature / K:", round(atoms.get_temperature(), 1))
print("Mobile atoms:", len(atoms) - len(fixed_pt_indices))

## Convert the trajectory to an animated GIF

> **As you watch:** Choose a view that shows vertical and lateral water motion.
>
> **What to look for:** A fixed camera, canvas, and frame timing help you read motion consistently.


This cell turns the trajectory into a GIF in three stages. ASE renders each frame to a PNG at a chosen view angle. Pillow adds a white background and gives every frame the same resolution. Imageio combines the frames into a looping GIF, then removes the temporary PNG files. `IPImage` displays the animation in the notebook.


In [ ]:
traj_path = "Model.traj"
rotation = "-90x,90y,0z"
trajectory = read(traj_path, index=":")
if animation_mode == "animated":
    png_files = []
    for i, frame in enumerate(trajectory):
        filename = f"frame_{i:04d}.png"
        write(filename, frame, show_unit_cell=0, rotation=rotation)
        png_files.append(filename)
    images = []
    for filename in png_files:
        with Image.open(filename) as image:
            image = image.convert("RGBA")
            background = Image.new("RGBA", image.size, (255, 255, 255, 255))
            images.append(Image.alpha_composite(background, image).convert("RGB"))
    imageio.mimsave("md.gif", images, duration=5, loop=0)
    for filename in png_files:
        os.remove(filename)
    display(IPImage("md.gif"))
else:
    write("md_static.png", trajectory[0], show_unit_cell=0, rotation=rotation)
    display(IPImage("md_static.png"))

## Student exercise: Interpret a molecular dynamics animation

Calculate the physical duration from the time step and number of steps. Inspect the trajectory and identify the fixed Pt region and mobile water molecules. Explain why GIF playback time differs from simulation time. Name two numerical checks for a model of an operating interface.

Record one prediction, one numerical or graphical result, and your chemical interpretation. Use copies of existing values so the main calculation stays intact.


In [ ]:
# Your code here.
# Use a separate variable for your changed parameter and display the comparison.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Attach a pretrained MACE calculator to an ASE interface model.
- Choose and record constraints, integration settings, and thermal initialization.
- Save and reload a trajectory, then convert its frames into an animation.
- Explain why a visual check needs energy, temperature, and structural checks.


### Questions for discussion

1. How do the number of integration steps and the time step set physical duration? How does GIF duration differ?
2. What evidence would you add before you report a water structure or transport property from this trajectory?
